In [3]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("week5") \
    .getOrCreate()


Spark version: 3.5.5


25/03/07 23:18:25 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Question 1

In [5]:
print("Spark version:", spark.version)

Spark version: 3.5.5


## Question 2

In [2]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet


--2025-03-07 23:17:38--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
18.172.223.120, 18.172.223.91, 18.172.223.180, ...rychx.cloudfront.net)... 
connected. to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.223.120|:443... 
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  64.3MB/s    in 1.0s    

2025-03-07 23:17:39 (64.3 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [4]:
df = spark.read.parquet("yellow_tripdata_2024-10.parquet")

In [6]:
df.printSchema()
df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [7]:
# Repartition to 4 partitions
df_repartitioned = df.repartition(4)

# Save to Parquet
output_path = "yellow_tripdata_2024-10-repartitioned"
df_repartitioned.write.mode("overwrite").parquet(output_path)


In [8]:
import os

# List all .parquet files in the output directory
parquet_files = [f for f in os.listdir(output_path) if f.endswith(".parquet")]

# Compute total size of all parquet files
total_size = sum(os.path.getsize(os.path.join(output_path, f)) for f in parquet_files)

# Compute the average file size in MB
average_size_mb = (total_size / len(parquet_files)) / (1024 * 1024)

print(f"Average Parquet file size: {average_size_mb:.2f} MB")


Average Parquet file size: 22.40 MB


## Question 3

In [10]:
from pyspark.sql.functions import date_format, col, to_date

# Convert pickup datetime to just the date (YYYY-MM-DD)
df_filtered = df.filter(date_format(col("tpep_pickup_datetime"), "yyyy-MM-dd") == "2024-10-15")

# Count the number of trips
trip_count = df_filtered.count()

print(f"Number of trips on October 15th: {trip_count}")


[Stage 6:============================================>              (6 + 2) / 8]

Number of trips on October 15th: 128893


## Question 4

In [11]:
from pyspark.sql.functions import col, unix_timestamp

# Compute trip duration in seconds
df_with_duration = df.withColumn("trip_duration_hours",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600
)

# Find the maximum trip duration
max_duration = df_with_duration.agg({"trip_duration_hours": "max"}).collect()[0][0]

print(f"Longest trip duration in hours: {max_duration:.2f}")


[Stage 9:=============================>                             (4 + 4) / 8]

Longest trip duration in hours: 162.62


In [12]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-07 23:32:19--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
18.172.223.180, 18.172.223.91, 18.172.223.120, ...rychx.cloudfront.net)... 
connected. to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.223.180|:443... 
200 OKequest sent, awaiting response... 
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-07 23:32:21 (50.7 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



## Question 6

In [13]:
# Load the taxi zone lookup data
df_zones = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")

# Load the Yellow Taxi October 2024 data
df_trips = spark.read.parquet("yellow_tripdata_2024-10.parquet")

# Register as temporary views for SQL queries
df_zones.createOrReplaceTempView("zones")
df_trips.createOrReplaceTempView("trips")


In [15]:
query = """
SELECT z.Zone, COUNT(t.PULocationID) AS pickup_count
FROM trips t
JOIN zones z ON t.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY pickup_count ASC
LIMIT 1
"""

least_frequent_zone = spark.sql(query)
least_frequent_zone.show(truncate=False)


[Stage 19:=============================>                            (4 + 4) / 8]

+---------------------------------------------+------------+
|Zone                                         |pickup_count|
+---------------------------------------------+------------+
|Governor's Island/Ellis Island/Liberty Island|1           |
+---------------------------------------------+------------+

